In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import keras
import keras_nlp, keras_hub
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

2026-04-05 06:31:54.943520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775370715.125832      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775370715.179083      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775370715.601141      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775370715.601177      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775370715.601180      24 computation_placer.cc:177] computation placer alr

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv", index_col="id")
test_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

In [3]:
# Create a new column of combined keyword and text

def combine_text(row):
    keyword = row["keyword"]
    text = row["text"]
    if isinstance(keyword, float):  # catches NaN
        keyword = "no_keyword"
    if isinstance(text, float):
        text = ""
    return f"keyword: {keyword}. tweet: {text}"

# train_df["model_text"].isna().sum()
train_df["model_text"] = train_df.apply(combine_text, axis=1)

In [4]:
import tensorflow as tf
data = tf.data.Dataset.from_tensor_slices(
    train_df["model_text"].astype(str).values
)


vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
    data,
    vocabulary_size = 20_000,
    lowercase=True,
    reserved_tokens=["[PAD]", "[CLS]", "[SEP]", "[UNK]", "[MASK]"]
)

I0000 00:00:1775370737.678829      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [5]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=128,     # good for tweets
    lowercase=True,
)

In [6]:
preprocessor = keras_hub.models.TextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased",
    sequence_length=128,
)

classifier = keras_hub.models.BertClassifier.from_preset(
    "bert_base_en_uncased",
    num_classes=1,
)


classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy()],
)

classifier.fit(
    train_df["text"],
    train_df["target"],
    validation_split=0.2,
    epochs=3,
    batch_size=8
)

Epoch 1/3


I0000 00:00:1775370840.100657      66 service.cc:152] XLA service 0x7a77fc026ce0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775370840.100698      66 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775370845.566878      66 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775370880.448995      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


762/762 ━━━━━━━━━━━━━━━━━━━━ 434s 477ms/step - binary_accuracy: 0.7276 - loss: 0.5496 - val_binary_accuracy: 0.8313 - val_loss: 0.4097
Epoch 2/3
762/762 ━━━━━━━━━━━━━━━━━━━━ 317s 416ms/step - binary_accuracy: 0.8503 - loss: 0.3740 - val_binary_accuracy: 0.8326 - val_loss: 0.3815
Epoch 3/3
762/762 ━━━━━━━━━━━━━━━━━━━━ 317s 416ms/step - binary_accuracy: 0.8818 - loss: 0.2835 - val_binary_accuracy: 0.8313 - val_loss: 0.4873


In [7]:
test_probs = classifier.predict(test_df["text"])
test_preds = (test_probs > 0.5).astype(int).reshape(-1)

102/102 ━━━━━━━━━━━━━━━━━━━━ 56s 503ms/step


In [8]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)